In [1]:
import os
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.padding import PKCS7
from cryptography.hazmat.backends import default_backend

## 1. Symmetric Layer (AES-CBC) for large files.

Here the biomedical data is encrypted using the **Advanced Encryption Standard (AES)** in **Cipher Block Chaining (CBC)** mode.

AES is a symmetric encryption algorithm, meaning the same key is used for both encryption and decryption.

Since AES operates on fixed-size blocks, the data must be padded using PKCS7 padding. Additionally, a random **Initialization Vector (IV)** is generated to ensure that identical plaintexts produce different ciphertexts, improving security.

In [16]:
def encrypt_data(data, aes_key):
    iv = os.urandom(16)

    padder = PKCS7(128).padder()
    padded_data = padder.update(data) + padder.finalize()

    cipher = Cipher(
        algorithms.AES(aes_key),
        modes.CBC(iv),
        backend=default_backend()
    )
    encryptor = cipher.encryptor()
    ciphertext = encryptor.update(padded_data) + encryptor.finalize()

    return iv, ciphertext

def decrypt_data(iv, ciphertext, aes_key):
    cipher = Cipher(
        algorithms.AES(aes_key),
        modes.CBC(iv),
        backend=default_backend()
    )
    decryptor = cipher.decryptor()
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()

    unpadder = PKCS7(128).unpadder()
    data = unpadder.update(padded_data) + unpadder.finalize()

    return data

We can use some sample data and try to encrypt/decrypt it.

In [17]:
aes_key = os.urandom(32)
raw_data = b"HowAreYouDoing?"
iv, ciphertext = encrypt_data(raw_data, aes_key)
print(f"Before encryption: {raw_data}")
print(f"After encryption: iv={iv} ciphertext={ciphertext}")

decrypt_data = decrypt_data(iv, ciphertext, aes_key)
print(f"After decryption: {decrypt_data}")

Before encryption: b'HowAreYouDoing?'
After encryption: iv=b'\xc5\xccHb\xf9\xcbN,d\xa9[:+2U\x1d' ciphertext=b'\x93t\x992\xca\t\x898\tq\x9f\x1e\x06\xbeb\x81'
After decryption: b'HowAreYouDoing?'


## 2. Assymetric layer (RSA-OAEP)

In this step, a randomly generated AES session key is encrypted using RSA with **Optimal Asymmetric Encryption Padding (OAEP)**.

This process is often referred to as “key wrapping” or creating a secure “envelope.” The AES key is protected using the recipient’s public key, ensuring that only the holder of the corresponding private key can recover it.

This hybrid approach combines the efficiency of symmetric encryption (AES) with the security of asymmetric encryption (RSA).

### 2.1 Generate RSA Keys

We use two types of keys in RSA (asymmetric).

**Private key**: Used for encrypting sensitive data.

**Public key**: Used for decryption.

In [18]:
def generate_rsa_keys():
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )
    public_key = private_key.public_key()
    return private_key, public_key

The keys generated are objects.

In [19]:
pub_key, pri_key = generate_rsa_keys()
print(pub_key)
print(pri_key)

### 2.2 Wrap using RSA-OAEP

In [20]:
def wrap_key(aes_key, public_key):
    encrypted_key = public_key.encrypt(
        aes_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return encrypted_key


def unwrap_key(encrypted_key, private_key):
    aes_key = private_key.decrypt(
        encrypted_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return aes_key

We can try to wrap/unwrap an AES key using public and private RSA keys.

In [21]:
aes_key = os.urandom(32)
private, public = generate_rsa_keys()

encrypted_key = wrap_key(aes_key,public)
decrypted_key = unwrap_key(encrypted_key,private)

print(f"Initial AES key\n{aes_key}\n\nEncrypted key\n{encrypted_key}\n\nDecrypted key\n{decrypted_key}")

Initial AES key
b'J\xfe\xec\xa1@\xd7U\x98Zk\xcd\xaf\x8b\x9d\xc3\x86\xb2\xaf\x8c4\xdfb\xeb\x93;\xedJ\x15\xb7Q\xa7\xdd'

Encrypted key
b"l\x89\x05p{G-\xee\x15{\xf9\xa6\xa7\x05\xaep4DM`\xc1\xa9`\xdc\xb7\x0b\xc8\xeat\xdd\xcb1\x9d\xa6\xc1\x80\x94W\x86e\xef]N7+\x08\x1a\xabh\x82v\x88`\x9e\xa7\x86\xa7\xfe[\xf4\xb6\xff\xf2\xe56>\x80*F\x8b\x93C\xc4!*\xddH\xec\xc2\x16\xb32\x90\x0b.e\xcb.\xfd\x07\xd6\x02C\xdb\xdew8FP\xc9\x15OS,\xad#\xaa't\xa1g\xb4uM\x9cG\xb4N\x91u\xa0\xa7\xaa;\x05\x808\x1c\xd7\xe7\x0fN\x82-B\r\xb4\xca\x9f\x02\xde\x1c3\x17;`\x1a\x9a{1\xc1w\x16\xaaYQ@\xf2\xbaW'\xd0\xe7\xa0h\x98\xd61\xbdj\x0b<\xb4\xa04Z\xf5\xcc\x1d\xd9\xa2\xcdH]KFa\xfe\x10\xab\xa6\xf8x\x8f\xd1\x8eu2\xdd\x14\xd2\xf7\x1a.J)\xe5\x8e`\xdf\xee\x95\xb4\xa3\x98\xa7j\x9aC\x9c\x107\x9e\xc1\x8a\x93\x00\xa7\xd6\xb0\r9\xef\x14\xa9\x88'\xfb\x9e}P)\xfb\x07D,&\x1c'\xc7\x93\xa0\xf7\x19\\\xfc"

Decrypted key
b'J\xfe\xec\xa1@\xd7U\x98Zk\xcd\xaf\x8b\x9d\xc3\x86\xb2\xaf\x8c4\xdfb\xeb\x93;\xedJ\x15\xb7Q\xa7\xdd'


## 4. Integrity layer (RSA-PSS).

To ensure data integrity and authenticity, the complete encrypted package is digitally signed using RSA with **Probabilistic Signature Scheme (PSS)**.

The signature is generated over the combined data (wrapped AES key, IV, and ciphertext). Any modification to the data will invalidate the signature.

This mechanism guarantees:
- **Integrity**: The data has not been altered.
- **Authenticity**: The data originates from a trusted source.

In [22]:
def sign_package(private_key, data):
    signature = private_key.sign(
        data,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    return signature


def verify_signature(public_key, data, signature):
    try:
        public_key.verify(
            signature,
            data,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        return True
    except:
        return False

In [23]:
aes_key = os.urandom(32)
raw_data = b"minMegetStaerkeKode"
corrputed_data = b"blablabla"

signature = sign_package(private, raw_data)

verification_no_errors = verify_signature(public, raw_data, signature)
print(verification_no_errors)
verification_corrupted_data = verify_signature(public, raw_data, corrputed_data)
print(verification_corrupted_data)

True
False


## 5. Data Vault Process

In this step, all components are combined into a secure “data vault.” The vault includes:
- The RSA-encrypted AES key
- The AES initialization vector (IV)
- The encrypted biomedical data (ciphertext)
- The digital signature

When opening the vault, the system first verifies the signature to ensure integrity before performing any decryption. If verification succeeds, the AES key is recovered using the private RSA key, and the original data is decrypted.

This layered approach ensures **confidentiality, integrity, and authenticity**, fulfilling the core security requirements of the system.

In [24]:
def create_vault(data, public_key, signing_private_key):
    # Generate AES session key
    aes_key = os.urandom(32)

    # Encrypt data
    iv, ciphertext = encrypt_data(data, aes_key)

    # Wrap AES key
    wrapped_key = wrap_key(aes_key, public_key)

    # Combine everything
    package = wrapped_key + iv + ciphertext

    # Sign package
    signature = sign_package(signing_private_key, package)

    return wrapped_key, iv, ciphertext, signature


def open_vault(wrapped_key, iv, ciphertext, signature, private_key, verify_public_key):
    package = wrapped_key + iv + ciphertext

    # Verify integrity FIRST
    if not verify_signature(verify_public_key, package, signature):
        raise Exception("❌ Integrity check failed! Data corrupted or tampered.")

    print("✅ Signature verified.")

    # Unwrap AES key
    aes_key = unwrap_key(wrapped_key, private_key)

    # Decrypt data
    data = decrypt_data(iv, ciphertext, aes_key)

    return data

## Example

In [26]:
# Simulated biomedical data (ECG)
biomedical_data = b"[72, 75, 78, 80, 76, 74]"

# Generate keys
receiver_private, receiver_public = generate_rsa_keys()
signer_private, signer_public = generate_rsa_keys()

# Create vault
wrapped_key, iv, ciphertext, signature = create_vault(
biomedical_data,
receiver_public,
signer_private
)

print("🔒 Data stored securely.")

# Open vault
decrypted_data = open_vault(
    wrapped_key,
    iv,
    ciphertext,
    signature,
    receiver_private,
    signer_public
)

print("🔓 Decrypted Data:", decrypted_data)

🔒 Data stored securely.
✅ Signature verified.


TypeError: 'bytes' object is not callable